In [61]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.text import text_to_word_sequence
from keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, Dropout
from keras.utils import to_categorical





In [62]:

df = pd.read_csv("text_emotion.csv")
print(df.head())
print(df.columns)


     tweet_id   sentiment       author  \
0  1956967341       empty   xoshayzers   
1  1956967666     sadness    wannamama   
2  1956967696     sadness    coolfunky   
3  1956967789  enthusiasm  czareaquino   
4  1956968416     neutral    xkilljoyx   

                                             content  
0  @tiffanylue i know  i was listenin to bad habi...  
1  Layin n bed with a headache  ughhhh...waitin o...  
2                Funeral ceremony...gloomy friday...  
3               wants to hang out with friends SOON!  
4  @dannycastillo We want to trade with someone w...  
Index(['tweet_id', 'sentiment', 'author', 'content'], dtype='object')


In [63]:
texts = df["content"].astype(str).tolist()
labels = df["sentiment"].tolist()

word_sequences = [text_to_word_sequence(text) for text in texts]
tokenizer = Tokenizer()
tokenizer.fit_on_texts(word_sequences)
tokens = tokenizer.word_index
vocabulary_size = len(tokens) + 1
print(vocabulary_size)
sequences = tokenizer.texts_to_sequences(texts)
max_length = max(len(seq) for seq in sequences)
print("Maximum sequence length:", max_length)


48998
Maximum sequence length: 37


In [64]:

padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)
X = padded_sequences
num_classes = len(le.classes_)
y = to_categorical(labels_encoded, num_classes=num_classes)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)


In [65]:
model = Sequential()

# Embedding layer
model.add(Embedding(
    input_dim=vocabulary_size,      # vocabulary size
    output_dim=10,             # embedding dimension
    input_length=max_length    # maximum sequence length
))

# First LSTM layer (128 units)
model.add(LSTM(128, return_sequences=True))

# Second LSTM layer (64 units)
model.add(LSTM(64))

# Fully connected layer (100 units, relu)
model.add(Dense(100, activation='relu'))

# Dropout layer (rate = 0.5)
model.add(Dropout(0.5))

# Output layer (num_classes units, softmax)
model.add(Dense(num_classes, activation='softmax'))

# Compile the model
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

c:\Users\clair\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [66]:
model.fit(X_train,y_train,batch_size=256,epochs=10)
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

Epoch 1/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 11s 78ms/step - accuracy: 0.2096 - loss: 2.2410
Epoch 2/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 9s 78ms/step - accuracy: 0.2453 - loss: 2.1646
Epoch 3/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 11s 104ms/step - accuracy: 0.3004 - loss: 2.0775
Epoch 4/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 14s 122ms/step - accuracy: 0.3386 - loss: 1.9747
Epoch 5/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 13s 122ms/step - accuracy: 0.3809 - loss: 1.8353
Epoch 6/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 13s 118ms/step - accuracy: 0.4273 - loss: 1.7091
Epoch 7/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 109ms/step - accuracy: 0.4630 - loss: 1.6216
Epoch 8/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 112ms/step - accuracy: 0.4993 - loss: 1.5278
Epoch 9/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 105ms/step - accuracy: 0.5305 - loss: 1.4353
Epoch 10/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 106ms/step - accuracy: 0.5533 - loss: 1.3707
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.2259 - loss: 2.6281
Test Accuracy: 0.22591666877269745
